# 07 — Leave-one-individual-out evaluation

This notebook evaluates split sensitivity by holding out each large-felid individual once.
It retrains the model for each held-out individual, so it is the heaviest notebook
and is best run overnight.

The algorithm is not re-selected using test performance. Each fold uses the same
expanded probabilistic workflow and reports aggregate held-out-individual performance.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction.timing import TimerLog, status
from carnivore_reconstruction.formal_large_felid import read_table_auto, summarize_with_linear, save_metric_bundle, prepare_formal_task_tables, balanced_sample_task_table
from carnivore_reconstruction.proposed import paper_metrics_for_reporting

MODEL_DIR = PROJECT_ROOT / "outputs" / "pretrained_model"
BASE_MODEL_PATH = MODEL_DIR / "pretrained_model.joblib"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "leave_one_individual_out"
FOLD_DIR = OUTPUT_DIR / "folds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

MAX_FOLDS = 8             # capped transfer/LOIO evaluation for practical runtime
MAX_TEST_TASKS_PER_FOLD = 100
MAX_VALIDATION_SUPPORT_TASKS_PER_FOLD = 120
MAX_TRAIN_TASKS_PER_FOLD = 2500
RANDOM_SEED = 42
KEEP_SCORED = False
VALIDATION_SPLIT = "validation"
TEST_SPLIT = "test"

timer = TimerLog()
from carnivore_reconstruction.utils import stable_hash01
import copy

In [ ]:
from carnivore_reconstruction import MotifReconstructionModel, tasks_from_tables

status("LOIO 1/5: loading formal task table and base config")
with timer.step("load_formal_tables"):
    base_model = MotifReconstructionModel.load(BASE_MODEL_PATH)
    base_config = base_model.config
    base_config.n_jobs = int(os.environ.get("CARNIVORE_N_JOBS", max(1, (os.cpu_count() or 2) - 1)))
    base_config.parallel_threshold_tasks = 1
    base_config.evaluate_environmental_exposure = False
    base_config.save_candidate_paths = False
    task_table = read_table_auto(MODEL_DIR / "task_table.parquet")
    task_points = read_table_auto(MODEL_DIR / "task_points.parquet")
    split_table_path = MODEL_DIR / "formal_individual_split_table.csv"
    split_table = pd.read_csv(split_table_path) if split_table_path.exists() else None
    task_table, task_points, split_table = prepare_formal_task_tables(
        task_table,
        task_points,
        split_table,
        seed=20260625,
        label="formal_task_table",
    )

if "formal_individual_key" not in task_table.columns:
    task_table["formal_individual_key"] = task_table[["dataset", "taxon", "animal_id"]].astype(str).agg("|".join, axis=1)

fold_keys = sorted(task_table["formal_individual_key"].dropna().astype(str).unique())
if MAX_FOLDS is not None:
    fold_keys = fold_keys[:int(MAX_FOLDS)]

print(f"LOIO folds: {len(fold_keys)}")
display(task_table.drop_duplicates("formal_individual_key")[["formal_individual_key", "dataset", "taxon", "animal_id"]].sort_values(["taxon", "formal_individual_key"]))

In [ ]:
from carnivore_reconstruction.benchmark import run_accuracy_benchmark

all_metric_parts = []
all_candidate_parts = []
all_runtime_parts = []
fold_summary_rows = []

status("LOIO 2/5: running folds")
for fold_i, heldout_key in enumerate(fold_keys, start=1):
    fold_name = f"fold_{fold_i:02d}_{heldout_key.replace('|', '_').replace('/', '-')}"
    fold_out = FOLD_DIR / fold_name
    fold_out.mkdir(parents=True, exist_ok=True)
    status(f"LOIO fold {fold_i}/{len(fold_keys)}: held out {heldout_key}")

    fold_table = task_table.copy()
    fold_table["loio_outer_fold"] = fold_name
    fold_table["loio_heldout_individual_key"] = heldout_key

    # Outer test is the held-out individual.  For validation support, keep the
    # original formal validation split among non-held-out individuals.
    fold_table["split"] = np.where(fold_table["formal_individual_key"].astype(str).eq(heldout_key), TEST_SPLIT, "train")
    original_validation = task_table["split"].astype(str).eq(VALIDATION_SPLIT) & ~task_table["formal_individual_key"].astype(str).eq(heldout_key)
    fold_table.loc[original_validation, "split"] = VALIDATION_SPLIT

    # If a fold has no validation support, create deterministic validation rows
    # from non-held-out training animals. This is used only for selectors, not
    # for reporting the held-out individual.
    if not fold_table["split"].astype(str).eq(VALIDATION_SPLIT).any():
        train_keys = sorted(fold_table.loc[fold_table["split"].eq("train"), "formal_individual_key"].astype(str).unique())
        if len(train_keys) >= 2:
            val_key = train_keys[int(stable_hash01(heldout_key) * len(train_keys)) % len(train_keys)]
            fold_table.loc[fold_table["formal_individual_key"].astype(str).eq(val_key), "split"] = VALIDATION_SPLIT

    test_table = fold_table[fold_table["split"].eq(TEST_SPLIT)].copy().reset_index(drop=True)
    if test_table.empty:
        print(f"Skipping {heldout_key}: no test tasks")
        continue
    test_table = balanced_sample_task_table(test_table, MAX_TEST_TASKS_PER_FOLD, seed=RANDOM_SEED + fold_i, label=f"{fold_name}_test")

    validation_table = fold_table[fold_table["split"].eq(VALIDATION_SPLIT)].copy().reset_index(drop=True)
    validation_table = balanced_sample_task_table(validation_table, MAX_VALIDATION_SUPPORT_TASKS_PER_FOLD, seed=RANDOM_SEED + fold_i, label=f"{fold_name}_validation_support")
    evaluation_table = pd.concat([validation_table, test_table], ignore_index=True, sort=False).drop_duplicates("task_uid")
    train_table_for_fold = fold_table[fold_table["split"].eq("train")].copy().reset_index(drop=True)
    train_table_for_fold = balanced_sample_task_table(train_table_for_fold, MAX_TRAIN_TASKS_PER_FOLD, seed=RANDOM_SEED + fold_i, label=f"{fold_name}_train")
    fold_fit_table = pd.concat([train_table_for_fold, validation_table, test_table], ignore_index=True, sort=False).drop_duplicates("task_uid")
    train_tasks = tasks_from_tables(fold_fit_table, task_points)
    evaluation_tasks = tasks_from_tables(evaluation_table, task_points)
    test_uids = set(test_table["task_uid"].astype(str))

    fold_timer = TimerLog()
    with fold_timer.step("fit_loio_model", n_tasks=int(fold_table["split"].eq("train").sum()), fold=fold_name):
        fold_model = MotifReconstructionModel(copy.deepcopy(base_config))
        fold_model.config.n_jobs = base_config.n_jobs
        fold_model.config.parallel_threshold_tasks = 1
        fold_model.config.evaluate_environmental_exposure = False
        fold_model.config.save_candidate_paths = False
        fold_model.fit(train_tasks, task_table=fold_fit_table, train_split="train", timer=fold_timer)

    with fold_timer.step("benchmark_loio_fold", n_tasks=len(evaluation_tasks), fold=fold_name):
        result = run_accuracy_benchmark(fold_model, evaluation_tasks, keep_scored=KEEP_SCORED, task_table=fold_fit_table)

    metrics = result["task_metrics"]
    metrics = metrics[metrics["task_uid"].astype(str).isin(test_uids)].copy().reset_index(drop=True)
    metrics["loio_outer_fold"] = fold_name
    metrics["loio_heldout_individual_key"] = heldout_key
    all_metric_parts.append(metrics)

    candidates = result.get("selected_candidates", pd.DataFrame())
    if candidates is not None and not candidates.empty and "task_uid" in candidates.columns:
        candidates = candidates[candidates["task_uid"].astype(str).isin(test_uids)].copy()
        candidates["loio_outer_fold"] = fold_name
        candidates["loio_heldout_individual_key"] = heldout_key
        all_candidate_parts.append(candidates)

    runtime = result.get("runtime", pd.DataFrame())
    if runtime is not None and not runtime.empty and "task_uid" in runtime.columns:
        runtime = runtime[runtime["task_uid"].astype(str).isin(test_uids)].copy()
        runtime["loio_outer_fold"] = fold_name
        runtime["loio_heldout_individual_key"] = heldout_key
        all_runtime_parts.append(runtime)

    metrics.to_csv(fold_out / "fold_task_metrics.csv", index=False)
    fold_timer.save(fold_out / "fold_runtime.csv")

    fold_meta = test_table.drop_duplicates("formal_individual_key").iloc[0].to_dict()
    fold_summary_rows.append({
        "loio_outer_fold": fold_name,
        "heldout_individual_key": heldout_key,
        "taxon": fold_meta.get("taxon"),
        "animal_id": fold_meta.get("animal_id"),
        "dataset": fold_meta.get("dataset"),
        "n_test_tasks": int(len(test_table)),
        "n_validation_support_tasks": int(len(validation_table)),
        "n_eval_tasks": int(len(evaluation_table)),
    })

print("Finished LOIO folds:", len(all_metric_parts))

In [ ]:
status("LOIO 3/5: aggregating fold outputs")
loio_task_metrics = pd.concat(all_metric_parts, ignore_index=True, sort=False) if all_metric_parts else pd.DataFrame()
loio_candidates = pd.concat(all_candidate_parts, ignore_index=True, sort=False) if all_candidate_parts else pd.DataFrame()
loio_runtime = pd.concat(all_runtime_parts, ignore_index=True, sort=False) if all_runtime_parts else pd.DataFrame()
loio_fold_summary = pd.DataFrame(fold_summary_rows)

loio_task_metrics.to_csv(OUTPUT_DIR / "loio_task_metrics.csv", index=False)
loio_candidates.to_csv(OUTPUT_DIR / "loio_selected_candidates.csv", index=False)
loio_runtime.to_csv(OUTPUT_DIR / "loio_runtime.csv", index=False)
loio_fold_summary.to_csv(OUTPUT_DIR / "loio_fold_summary.csv", index=False)

print("loio_task_metrics:", loio_task_metrics.shape)
display(loio_fold_summary.head())

In [ ]:
status("LOIO 4/5: saving LOIO summaries")
with timer.step("save_loio_summaries", n_tasks=loio_task_metrics["task_uid"].nunique() if not loio_task_metrics.empty else 0):
    if loio_task_metrics.empty:
        raise ValueError("No LOIO metrics were produced.")
    save_metric_bundle(loio_task_metrics, OUTPUT_DIR, "loio", paper_filter_func=paper_metrics_for_reporting)

    # A fold-level summary helps show whether the method is stable across held-out individuals.
    paper_loio = paper_metrics_for_reporting(loio_task_metrics)
    fold_method_summary = summarize_with_linear(paper_loio, ["loio_heldout_individual_key", "taxon", "method"])
    fold_method_summary.to_csv(OUTPUT_DIR / "paper_loio_by_heldout_individual_method_summary.csv", index=False)

    timer.save(OUTPUT_DIR / "runtime_loio_overall.csv")

display(pd.read_csv(OUTPUT_DIR / "paper_loio_method_summary.csv"))

In [ ]:
status("LOIO 5/5: done")
print("Saved outputs to:", OUTPUT_DIR)